# (Optional) Deploy Legacy MCP Lambda (legacy)

`01_deploy_mcp_lambda.ipynb`에서 배포한 `fhir-mcp-server` (with_metadata)에 추가로,
비교 데모용 `fhir-mcp-legacy` Lambda를 배포합니다.

동일한 `mcp/` 소스를 사용하며 `MCP_MODE=legacy` 환경변수만 다르게 설정합니다.

## Step 1: Configuration

In [ ]:
import boto3, json, time, os, zipfile, shutil, subprocess

REGION = boto3.session.Session().region_name or os.environ.get("AWS_DEFAULT_REGION") or os.environ.get("AWS_REGION")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

emr_client = boto3.client("emr-serverless", region_name=REGION)
apps = emr_client.list_applications()["applications"]
emr_app = next(a for a in apps if a["name"] == "fhir-data-query")
EMR_APPLICATION_ID = emr_app["id"]

iam_client = boto3.client("iam")
roles = iam_client.list_roles(MaxItems=200)["Roles"]
emr_exec_role = next(r for r in roles if "emr-serverless-execution-role" in r["RoleName"])
EMR_EXECUTION_ROLE_ARN = emr_exec_role["Arn"]

lambda_client = boto3.client("lambda", region_name=REGION)

FUNC_NAME = "fhir-mcp-legacy"
MCP_DIR = "../mcp"

print(f"Account: {ACCOUNT_ID}, Region: {REGION}")
print(f"EMR Application ID: {EMR_APPLICATION_ID}")


## Step 2: Create Lambda Execution Role

In [ ]:
LAMBDA_ROLE_NAME = "fhir-mcp-legacy-lambda-role"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["emr-serverless:AccessLivyEndpoints", "emr-serverless:GetApplication"],
         "Resource": f"arn:aws:emr-serverless:{REGION}:{ACCOUNT_ID}:/applications/{EMR_APPLICATION_ID}"},
        {"Effect": "Allow", "Action": "iam:PassRole", "Resource": EMR_EXECUTION_ROLE_ARN,
         "Condition": {"StringLike": {"iam:PassedToService": "emr-serverless.amazonaws.com"}}},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
         "Resource": "arn:aws:logs:*:*:*"},
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:ListBucket"],
         "Resource": [f"arn:aws:s3:::fhir-data-{ACCOUNT_ID}-{REGION}",
                      f"arn:aws:s3:::fhir-data-{ACCOUNT_ID}-{REGION}/metadata/*"]}
    ]
}

try:
    iam_client.create_role(RoleName=LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    print(f"Created role: {LAMBDA_ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"Role already exists: {LAMBDA_ROLE_NAME}")

iam_client.put_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyName="EMRServerlessLivyAccess",
                           PolicyDocument=json.dumps(inline_policy))
LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{LAMBDA_ROLE_NAME}"
print(f"Role ARN: {LAMBDA_ROLE_ARN}")
time.sleep(10)


## Step 3: Package & Deploy Lambda

In [7]:
env_vars = {
    "EMR_APPLICATION_ID": EMR_APPLICATION_ID,
    "EMR_EXECUTION_ROLE_ARN": EMR_EXECUTION_ROLE_ARN,
    "AWS_REGION_NAME": REGION,
    "AWS_ACCOUNT_ID": ACCOUNT_ID,
    "MCP_MODE": "legacy",
}

build_dir = f"/tmp/{FUNC_NAME}-build"
zip_path = f"/tmp/{FUNC_NAME}.zip"
if os.path.exists(build_dir):
    shutil.rmtree(build_dir)
os.makedirs(build_dir)

subprocess.run(["pip", "install", "requests", "-t", build_dir, "--quiet"], check=True)
for item in ["handler.py", "emr_client.py", "metadata_loader.py"]:
    shutil.copy2(os.path.join(MCP_DIR, item), build_dir)
shutil.copytree(os.path.join(MCP_DIR, "tools"), os.path.join(build_dir, "tools"), dirs_exist_ok=True)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(build_dir):
        for f in files:
            full = os.path.join(root, f)
            zf.write(full, os.path.relpath(full, build_dir))

print(f"Package: {os.path.getsize(zip_path)/1024/1024:.1f} MB")

with open(zip_path, "rb") as f:
    zip_bytes = f.read()

try:
    resp = lambda_client.create_function(
        FunctionName=FUNC_NAME, Runtime="python3.13", Role=LAMBDA_ROLE_ARN,
        Handler="handler.handler", Code={"ZipFile": zip_bytes},
        Timeout=300, MemorySize=512, Environment={"Variables": env_vars},
    )
    print(f"Created: {resp['FunctionArn']}")
except lambda_client.exceptions.ResourceConflictException:
    resp = lambda_client.update_function_code(FunctionName=FUNC_NAME, ZipFile=zip_bytes)
    waiter = lambda_client.get_waiter("function_updated_v2")
    waiter.wait(FunctionName=FUNC_NAME)
    lambda_client.update_function_configuration(
        FunctionName=FUNC_NAME, Timeout=300, MemorySize=512,
        Environment={"Variables": env_vars},
    )
    print(f"Updated: {resp['FunctionArn']}")

print("\n✅ Lambda deployed!")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
botocore 1.42.83 requires urllib3<1.27,>=1.25.4; python_version < "3.10", but you have urllib3 2.6.3 which is incompatible.
awscli 2.33.15 requires prompt-toolkit<3.0.52,>=3.0.24, but you have prompt-toolkit 3.0.52 which is incompatible.
awscli 2.33.15 requires python-dateutil<=2.9.0,>=2.1, but you have python-dateutil 2.9.0.post0 which is incompatible.


Package: 0.9 MB
Updated: arn:aws:lambda:us-west-2:997924005000:function:fhir-mcp-legacy

✅ Lambda deployed!


## Step 4: Test Lambda

In [ ]:
test_event = {"toolName": "list_tables", "arguments": {}}
print("Invoking (may take 1-2 min for Livy session)...")
resp = lambda_client.invoke(FunctionName=FUNC_NAME, Payload=json.dumps(test_event))
result = json.loads(resp["Payload"].read())
print(json.dumps(result, indent=2, ensure_ascii=False)[:2000])


## ✅ Done!

`fhir-mcp-legacy` Lambda가 배포되었습니다 (`data_legacy` 네임스페이스, 메타데이터 없음).

다음: `06_setup_legacy_gateway.ipynb`에서 AgentCore Gateway를 설정합니다.